# 08 · The prediction API and model monitoring

**Use case:** the verified-purchase model (04) goes into an application: score a review the app just received
(inductive — from features, not a stored row), score in batches, and watch the model's inputs and outputs drift.

**Sub-tasks**
1. Read the model's feature schema (`predict.model`)
2. Inductive prediction from raw features
3. Batch prediction
4. Monitoring summary, thresholds and alert configuration
5. Recent prediction monitoring for the model

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST_SDK_2' with 17 scopes


In [2]:
p = get_or_create_project(ls, "verified", kind="data_science")
model = next(m for m in p.models.list() if m.get("metrics"))
print("model:", model["model_id"], model.get("model_type"), "· AUROC", model["metrics"].get("auroc"))

reusing project 28106708-9ec0-48e4-b292-d600b9bc2ed8 (amazon-reviews-verified, status=ready)
project ready: status=ready · type=data_science · files=['customer.csv', 'product.csv', 'review.csv']


model: 61b6e350-ea4c-4219-a694-00835a452fd5 Baseline · AUROC 0.6280805005443502


In [3]:
schema = ls.predict.model(model["model_id"])
print("task:", schema.get("task_type"), "· entity:", schema.get("entity_table"), schema.get("entity_col"), "· target:", schema.get("target_col"))
print("required features:", [f.get("name") if isinstance(f, dict) else f for f in (schema.get("required_features") or [])][:10])
print("optional features:", [f.get("name") if isinstance(f, dict) else f for f in (schema.get("optional_features") or [])][:10])

task: binary_classification · entity: review __row_id__ · target: verified
required features: ['rating', 'review_text', 'summary']
optional features: []


## Inductive: a review the platform has never stored

Pass the review's own columns plus `related_entities` — the ids of the rows it links to (`product`, `customer`). A relational model predicts from the neighbourhood, so it refuses (with a clear message) if you give it the entity's columns alone.

In [4]:
review = p.table("review").to_pandas()
sample = review.iloc[0].to_dict()
features = {"review_text": "Arrived on time, exactly as described. The kids loved it and we read it twice the first night.",
            "summary": "Great buy", "rating": 5}
out = ls.predict.predict(model_id=model["model_id"], mode="inductive", features=features,
                         related_entities={"product": sample["product_id"], "customer": sample["customer_id"]})
print("verified?", out["result"]["prediction"], "· probability", round(out["result"].get("probability") or 0, 3), "· mode", out["mode"])
if out.get("warnings"): print("warnings:", out["warnings"])

verified? 1 · probability 0.909 · mode inductive


In [5]:
top = ls.predict.top(model["model_id"], n=3)
ids = [r["entity_id"] for r in top["ranking"]]
batch = ls.predict.predict_batch(model_id=model["model_id"], entity_ids=ids)
print(batch["succeeded"], "scored;", [round(pr.get("probability") or 0, 3) for pr in batch["predictions"]])

3 scored; [0.922, 0.921, 0.92]


## Monitoring

Every prediction is logged; the monitoring endpoints summarise input drift and output distribution, and thresholds + alerts turn that into an email or a webhook.

In [6]:
summary = p.monitoring.summary()
print(json.dumps(summary, indent=2, default=str)[:1500])

{
  "state": "ready",
  "model_id": "61b6e350-ea4c-4219-a694-00835a452fd5",
  "project_name": "amazon-reviews-verified",
  "task_type": "binary_classification",
  "period_days": 7,
  "version": 1,
  "family": 0,
  "version_label": "",
  "is_active": true,
  "total_predictions": 18,
  "total_alerts": 0,
  "critical_alerts": 0,
  "features_stable": 0,
  "features_warning": 0,
  "features_critical": 0,
  "features_insufficient": 9,
  "has_baseline": true,
  "anomaly_score": 1.33,
  "features": [
    {
      "feature_name": "summary",
      "dtype": "text",
      "health": "insufficient_data",
      "source_table": "review",
      "has_baseline": true,
      "request_count": 18,
      "missing_count": 16,
      "null_count": 0,
      "type_error_count": 0,
      "range_violation_count": 0,
      "new_category_count": 0,
      "observed_mean": null,
      "observed_std": null,
      "training_mean": null,
      "training_std": null,
      "training_min": null,
      "training_max": null,
  

In [7]:
print("thresholds:", p.monitoring.thresholds())
print("alerts:", p.monitoring.alerts_config())
recent = ls.predict.monitoring(model["model_id"], days=7)
print(json.dumps(recent, indent=2, default=str)[:800])

thresholds: {'defaults': {'null_rate_warning': 0.1, 'null_rate_critical': 0.3, 'drift_warning': 1.0, 'drift_critical': 2.0, 'type_error_warning': 0.01, 'type_error_critical': 0.05, 'range_violation_warning': 0.1, 'range_violation_critical': 0.25, 'new_category_warning': 0.1, 'new_category_critical': 0.3, 'unseen_id_warning': 0.2, 'unseen_id_critical': 0.5, 'boolean_shift_warning': 0.15, 'boolean_shift_critical': 0.3, 'text_length_drift_warning': 1.0, 'text_length_drift_critical': 2.0, 'js_divergence_warning': 0.1, 'js_divergence_critical': 0.3, 'min_sample_distribution': 50}, 'overrides': {}, 'effective': {'null_rate_warning': 0.1, 'null_rate_critical': 0.3, 'drift_warning': 1.0, 'drift_critical': 2.0, 'type_error_warning': 0.01, 'type_error_critical': 0.05, 'range_violation_warning': 0.1, 'range_violation_critical': 0.25, 'new_category_warning': 0.1, 'new_category_critical': 0.3, 'unseen_id_warning': 0.2, 'unseen_id_critical': 0.5, 'boolean_shift_warning': 0.15, 'boolean_shift_critica

alerts: {'email_alerts_enabled': False, 'alert_email': None, 'webhook_enabled': False, 'webhook_url': None, 'webhook_secret': '', 'webhook_secret_set': False, 'min_alert_interval_hours': 24, 'min_severity': 'warning', 'anomaly_alert_enabled': False, 'anomaly_score_threshold': 3.0}
{
  "state": "ready",
  "model_id": "61b6e350-ea4c-4219-a694-00835a452fd5",
  "project_name": "amazon-reviews-verified",
  "task_type": "binary_classification",
  "period_days": 7,
  "version": 1,
  "family": 0,
  "version_label": "v1",
  "is_active": true,
  "total_predictions": 18,
  "total_alerts": 0,
  "critical_alerts": 0,
  "features_stable": 0,
  "features_warning": 0,
  "features_critical": 0,
  "features_insufficient": 9,
  "has_baseline": true,
  "anomaly_score": 1.33,
  "features": [
    {
      "feature_name": "summary",
      "dtype": "text",
      "health": "insufficient_data",
      "source_table": "review",
      "has_baseline": true,
      "request_count": 18,
      "missing_count": 16,
     

In [8]:
save_metrics(".", {"notebook": "08_predict_api_and_monitoring", "task": "serve + monitor the binary model from 04", "model": model.get("model_type"),
                   "project_id": p.id, "model_id": model["model_id"],
                   "headline": {"batch_scored": batch["succeeded"], "monitoring_keys": list(summary.keys())[:8] if isinstance(summary, dict) else None}})

wrote results/metrics.json


PosixPath('results/metrics.json')